In [2]:
import pandas as pd
from groq import Groq
from dotenv import load_dotenv
import os

load_dotenv()
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
client = Groq(api_key=GROQ_API_KEY)

print("AI model ready!")

AI model ready!


In [3]:
rfm = pd.read_csv("../data/processed/rfm_segments.csv")
forecast = pd.read_csv("../data/processed/demand_forecast.csv")
anomalies = pd.read_csv("../data/processed/seller_anomalies.csv")
anomalies = anomalies[anomalies['anomaly'] == -1]

# Build summary stats
total_customers   = len(rfm)
champions         = (rfm['Segment'] == 'Champions').sum()
at_risk           = (rfm['Segment'] == 'At Risk').sum()
churned           = rfm['Churned'].sum()
anomaly_count     = len(anomalies)
worst_delay       = anomalies['avg_delay'].max()
next_30_day_avg   = forecast.tail(30)['yhat'].mean()

print(f"Customers: {total_customers:,}")
print(f"Champions: {champions:,}")
print(f"At Risk: {at_risk:,}")
print(f"Anomalies: {anomaly_count:,}")
print(f"Forecast next 30 days avg: {next_30_day_avg:.0f} orders/day")

Customers: 96,478
Champions: 7,781
At Risk: 39,822
Anomalies: 149
Forecast next 30 days avg: 301 orders/day


In [4]:



report_prompt = f"""
You are an Operations Intelligence AI for an e-commerce company.
Generate a professional weekly operations report based on this data:

CUSTOMER ANALYTICS:
- Total customers: {total_customers:,}
- Champions (best customers): {champions:,}
- At Risk customers: {at_risk:,}
- Churned customers: {churned:,}

DEMAND FORECAST:
- Expected orders next 30 days: {next_30_day_avg:.0f} per day

ANOMALY DETECTION:
- Anomalous sellers flagged: {anomaly_count:,}
- Worst delivery delay detected: {worst_delay:.1f} days

Write a 3-paragraph executive summary with:
1. Overall business health
2. Key risks and alerts
3. Recommended actions for operations team
"""

response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",  # Updated to a supported, active model
    messages=[{"role": "user", "content": report_prompt}]
)
print(response.choices[0].message.content)

**Executive Summary**

Our e-commerce platform's overall business health is a mixed bag, with both positive and concerning trends. On the positive side, we have a sizable customer base of 96,478, with a notable 7,781 champions who are our best customers. However, a closer look at the data reveals that 39,822 customers are at risk, and a substantial 68,837 have already churned. This suggests that while we have a loyal core, our customer retention efforts need significant improvement. In terms of demand, our forecast indicates a steady stream of 301 orders per day over the next 30 days, which is a positive sign for our revenue prospects.

Key risks and alerts are centered around customer churn and anomalies in our seller network. The high number of at-risk and churned customers (39,822 and 68,837, respectively) is a pressing concern that requires immediate attention. Furthermore, our anomaly detection systems have flagged 149 suspicious sellers, with one notable instance of a 35-day deli

In [5]:

email_prompt = f"""
You are an AI assistant for an e-commerce operations team.
Write a professional supplier reorder email based on this forecast:

- Product category: bed_bath_table (highest demand category)
- Current avg daily orders: {next_30_day_avg:.0f}
- Forecast shows demand will increase next 30 days
- Current stock risk: Medium

Write a concise, professional reorder email to the supplier.
Include: subject line, greeting, order details, urgency, closing.
"""

email_response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[{"role": "user", "content": email_prompt}]
)
print("=== AI GENERATED SUPPLIER EMAIL ===\n")
print(email_response.choices[0].message.content)

=== AI GENERATED SUPPLIER EMAIL ===

Subject: Urgent Reorder Request for Bed, Bath, and Table Category

Dear [Supplier's Name],

I hope this email finds you well. As we continue to experience high demand in the bed, bath, and table category, our current stock levels are at a medium risk of depletion. With an average of 301 daily orders and a forecast indicating increased demand over the next 30 days, we would like to place a reorder to ensure timely fulfillment of customer orders.

We kindly request that you expedite the shipment of the following products to replenish our stock:
- [List specific products or SKUs]
- [Quantity of each product]

Given the medium stock risk and anticipated increase in demand, we urge you to prioritize this order and provide us with an estimated delivery date as soon as possible. This will enable us to maintain optimal inventory levels and meet customer expectations.

Please confirm receipt of this request and provide us with an update on the expected shipm

In [6]:
with open('../outputs/weekly_report.txt', 'w') as f:
    f.write("=== SMARTOPS WEEKLY OPS REPORT ===\n\n")
    f.write(response.choices[0].message.content)

with open('../outputs/supplier_email.txt', 'w') as f:
    f.write("=== AI GENERATED SUPPLIER EMAIL ===\n\n")
    f.write(email_response.choices[0].message.content)

print("Module 4 Complete!")
print("Reports saved to outputs/ folder!")

Module 4 Complete!
Reports saved to outputs/ folder!


In [7]:
pip install secure-smtplib

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
import os

# Get project root folder
BASE_DIR = os.path.dirname(os.path.dirname(os.path.abspath('__file__')))
outputs_dir = os.path.join(BASE_DIR, "outputs")
os.makedirs(outputs_dir, exist_ok=True)

# Save weekly report
with open(os.path.join(outputs_dir, "weekly_report.txt"),
          "w", encoding="utf-8") as f:
    f.write(response.choices[0].message.content)

# Save supplier email
with open(os.path.join(outputs_dir, "supplier_email.txt"),
          "w", encoding="utf-8") as f:
    f.write(email_response.choices[0].message.content)

print("Files saved to:", outputs_dir)
print("Weekly report length:", 
      len(response.choices[0].message.content))
print("Supplier email length:", 
      len(email_response.choices[0].message.content))

Files saved to: c:\Users\abmin\AI-Powered-E-Commerce-Operations-Intelligence-System\outputs
Weekly report length: 2224
Supplier email length: 1217


In [11]:
import os
from dotenv import load_dotenv

load_dotenv()

print("BREVO_LOGIN =", os.getenv("BREVO_LOGIN"))
print("SENDER_EMAIL =", os.getenv("SENDER_EMAIL"))
print("BREVO_SMTP_KEY =", os.getenv("BREVO_SMTP_KEY"))

BREVO_LOGIN = None
SENDER_EMAIL = None
BREVO_SMTP_KEY = None


In [12]:
import os
from dotenv import load_dotenv
load_dotenv()
import smtplib
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart

def send_report_email(report_text, recipient_email):
    smtp_server     = "smtp-relay.brevo.com"
    port            = 587
    login_email     = os.getenv("BREVO_LOGIN")
    sender_email    = os.getenv("SENDER_EMAIL")
    sender_password = os.getenv("BREVO_SMTP_KEY")
    
    msg = MIMEMultipart()
    msg['From']    = sender_email
    msg['To']      = recipient_email
    msg['Subject'] = "SmartOps Weekly Operations Report"
    
    msg.attach(MIMEText(report_text, 'plain'))
    
    with smtplib.SMTP(smtp_server, port) as server:
        server.starttls()
        server.login(login_email, sender_password)
        server.sendmail(sender_email,
                       recipient_email,
                       msg.as_string())
    
    print(f"Report sent to {recipient_email}")
    
# 1. Grab both raw text outputs from your variables
summary_text = response.choices[0].message.content
supplier_text = email_response.choices[0].message.content

# 2. Extract the Supplier Email body and drop its original subject line
clean_supplier = supplier_text.replace("Subject: Urgent Reorder Request for Bed, Bath, and Table Products\n\n", "")

# 3. Cut off the supplier text right before "Best regards," 
# This completely drops the middle signature block so it won't clutter the center
supplier_body_only = clean_supplier.partition("Best regards,")[0].strip()

# 4. Define your active local or hosted Streamlit dashboard link
# Change "localhost:8501" to your production URL when you host it online!
streamlit_dashboard_url = "https://your-app.streamlit.app"
clean_supplier = supplier_text.replace(
    "Subject: Urgent Reorder Request for Bed, Bath, and Table Products\n\n", "")
supplier_body_only = clean_supplier.partition(
    "Best regards,")[0].strip()

payload = f"""
======================================================================
                 OUTBOUND SUPPLIER REORDER DRAFT
======================================================================
{supplier_body_only}


======================================================================
                  INTERNAL EXECUTIVE SUMMARY 
======================================================================
{summary_text}

======================================================================
📊 LIVE SMARTOPS DASHBOARD CALL-TO-ACTION
======================================================================
To inspect real-time anomaly trends or update inventory parameters:
👉 CLICK HERE TO OPEN DASHBOARD: {streamlit_dashboard_url}
======================================================================

======================================================================
Best regards,
Vinay
E-commerce Operations Team
======================================================================
"""

# 5. Fire the perfectly rearranged single payload to your inbox
#send_report_email(payload, "vinaygod100@gmail.com")
#print("✅ Rearranged email sequence dispatched successfully!")

print("✅ AI Report Generated Successfully!")
print(payload)

✅ AI Report Generated Successfully!

                 OUTBOUND SUPPLIER REORDER DRAFT
Subject: Urgent Reorder Request for Bed, Bath, and Table Category

Dear [Supplier's Name],

I hope this email finds you well. As we continue to experience high demand in the bed, bath, and table category, our current stock levels are at a medium risk of depletion. With an average of 301 daily orders and a forecast indicating increased demand over the next 30 days, we would like to place a reorder to ensure timely fulfillment of customer orders.

We kindly request that you expedite the shipment of the following products to replenish our stock:
- [List specific products or SKUs]
- [Quantity of each product]

Given the medium stock risk and anticipated increase in demand, we urge you to prioritize this order and provide us with an estimated delivery date as soon as possible. This will enable us to maintain optimal inventory levels and meet customer expectations.

Please confirm receipt of this request an